In [ ]:
%matplotlib widget

#Pyxem tends to give alot of warnings. We choose to ignore most of them in this notebook. However this is NOT recommended for your own work, as some warnings might give very useful information!
import warnings
warnings.filterwarnings('ignore')

import hyperspy.api as hs #General hyperspy package
import pyxem as pxm #Electron diffraction tools based on hyperspy
import numpy as np #General numerical and matrix support
import matplotlib.pyplot as plt #Plotting tools
import matplotlib.colors as mcolors #Some plotting color tools
from matplotlib.cm import ScalarMappable
import diffpy #Electron diffraction tools
import requests

import pandas as pd
import seaborn as sb
from tabulate import tabulate

#Import path handling tool
from pathlib import Path

#Import indexation and plotting tools
from diffsims.generators.rotation_list_generators import get_beam_directions_grid
from diffsims.libraries.structure_library import StructureLibrary
from diffsims.generators.diffraction_generator import DiffractionGenerator
from diffsims.generators.library_generator import DiffractionLibraryGenerator
from diffsims.generators.simulation_generator import SimulationGenerator

#Import orientation handling tools
from orix.quaternion import Rotation, symmetry, Orientation
from orix.crystal_map import Phase
from orix.vector.vector3d import Vector3d
from orix.vector import Miller
from orix.projections import StereographicProjection
from orix import plot
from orix.plot import IPFColorKeyTSL
from orix.sampling import get_sample_reduced_fundamental
from orix.io import save, load

: 

Load data

In [ ]:
datapath = Path("20260417_Ingrid/20260417_103611/SPED_CL8_256x256px_w10_3p40nm_ws_10ms_tx_m2p2_Ty3p8_cropped_centered_masked.hspy")

In [ ]:
lazy = True
signal = hs.load(str(datapath), lazy=lazy) #Load the data. Use `lazy=True` if you are experiencing memory issues
if lazy:
    signal.rechunk(nav_chunks=(32, 32),sig_chunks=(32, 32))

Create a library of diffraction patterns

In [ ]:
fname = 'GaAs.cif'
url = "https://materials.springer.com/downloads/track-required/true?path=%2Fassets%2Fsm_isp%2Fcrystallographic%2F311%2Fsm_isp_sd_0311662%2Fsm_isp_sd_0311662_download.cif&componentId=Download+Data+CIF"
r = requests.get(url)
open(fname , 'wb').write(r.content)

GaAs = Phase.from_cif(fname)

create simulations

In [ ]:
resolution = 0.5 #0.5. The angular spacing between each orientation

beam_directions = get_sample_reduced_fundamental(resolution, point_group=GaAs.point_group) #Get a uniform sampling of euler angles
#print(f'Simulated {beam_directions.size} beam directions')
beam_orientations = Orientation(beam_directions, symmetry=GaAs.point_group)
#beam_orientations.scatter('ipf')

In [ ]:
minimum_intensity = 1E-8 #Tuneable parameter
max_excitation_error = 1E-1 #Tuneable parameter

sim_gen = SimulationGenerator(
    accelerating_voltage=signal.metadata.Acquisition_instrument.TEM.beam_energy,
    scattering_params='lobato',
    precession_angle=1,
    shape_factor_model='linear',
    approximate_precession=True,
    minimum_intensity=minimum_intensity
) #Create a diffraction simulation generator

calibration = signal.axes_manager[-1].scale #Get the calibration from the signal
reciprocal_radius = np.max(np.abs(signal.axes_manager[-1].axis)) #Get the maximum reciprocal radius to simulate
half_shape = np.min(signal.axes_manager.signal_shape)//2 #Get the half-shape of the patterns

simulations = sim_gen.calculate_diffraction2d(
    phase=GaAs,
    rotation=beam_directions,
    reciprocal_radius=reciprocal_radius,
    with_direct_beam=False, #False
    max_excitation_error=max_excitation_error,
    shape_factor_width=None,
    debye_waller_factors=None    
)

In [ ]:


simulations.plot(interactive=True)

template match a subset

In [ ]:
# pull out a random image and simulation
nx, ny = signal.axes_manager.navigation_shape
n = 3 #The number of scan points to look at around the random point
np.random.seed(197405)#Set a seed for the random number generator - let us use the NORTEM project number
x, y = np.random.randint(nx-n), np.random.randint(ny-n)
subset = signal.inav[x:x+n, y:y+n]
try:
    subset.compute()
except Exception as e:
    pass


npt = int(np.floor(np.sqrt(np.sum((np.array(subset.axes_manager.signal_shape)//2)**2)))) #Number of radial points.
npt_azim = 360 #Number of azimuthal sampling points
subset.change_dtype('float32')
#subset.data = subset.data.astype(np.float32) # Added after "unsupported array-type appeared"
subset_pol = subset.get_azimuthal_integral2d(npt=npt, npt_azim=npt_azim)

In [ ]:
#n = simulations.current_size #Keep the results from matching all the templates
n = 8 #If you only want to keep the five best results
subset_results = subset_pol.get_orientation(simulation=simulations, n_best=n, frac_keep=1.0)
subset_results

In [ ]:
n = subset_results.num_rows
#n = 1 #If too crowded markers in the plot, uncomment this line to only plot the highest correlation scores

subset_pol.plot(norm='symlog')
subset_pol.add_marker(
    subset_results.to_single_phase_polar_markers(
        n_best=n,
        signal_axes=subset_pol.axes_manager.signal_axes
    )
)

In [ ]:
subset.plot(cmap='viridis_r', norm='symlog')
subset.add_marker(subset_results.to_markers(annotate=True))

In [ ]:
pixel = [0,0]
correlations_i = subset_results.inav[pixel].data[:, 1]
template_indices_i = (subset_results.inav[pixel].data[:,0]).astype('int16')
beam_orientations_i = beam_orientations[template_indices_i]
euler_angles_i = beam_orientations_i.to_euler()

fig = plt.figure()
ax = fig.add_subplot(111, projection="ipf", symmetry=GaAs.point_group)
ax.scatter(beam_orientations_i, c=correlations_i, cmap='inferno')
ax.set_title(f'Correlation for pixel {pixel}')

In [ ]:
#fmt = 'html'
#fmt = 'simple'
#fmt = 'latex'
fmt = 'pipe' #For PHP Markdown

# Get the simulation of the template index
beam_direction, phase_index, simulation = simulations.get_simulation(template_indices_i[0]) #Returns a tuple with the beam direction, the phase index, and the simulation
phi1, theta, phi2 = beam_direction.to_euler(degrees=True).flatten() #Get the euler angles of the beam direction follwing the Bunge convention used in Orix.

#Create a nicely formatted table of the hkl indices.
table = tabulate(simulation.hkl, headers=['h', 'k', 'l'], floatfmt='.0f', tablefmt=fmt) 
print(f"(hkl) indices of reflections of euler angles ({phi1:.1f}, {theta:.1f}, {phi2:.1f}) of phase number {phase_index}:\n{table}")

In [ ]:
g1 = Miller(hkl=simulation.hkl[0], phase=GaAs)
g2 = Miller(hkl=simulation.hkl[2], phase=GaAs)
P = g1.cross(g2).round()
print(f"Zone axis of template is close to [{P.u[0]:.0f} {P.v[0]:.0f} {P.w[0]:.0f}]")

In [ ]:
npt = int(np.floor(np.sqrt(np.sum((np.array(subset.axes_manager.signal_shape)//2)**2)))) #Number of radial points.
npt_azim = 360 #Number of azimuthal sampling points
signal.change_dtype('float32') #must be float
signal_pol = signal.get_azimuthal_integral2d(npt=npt, npt_azim=npt_azim)
if lazy:
    nav_chunks = (32, 32) # values from notebook
    sig_chuncks = (36, 30) #default values from notebook
    #signal_pol.rechunk(nav_chunks, sig_chunks)
    signal_pol.data = signal_pol.data.rechunk(nav_chunks + sig_chuncks)
    if True: # Set to False if you want to keep the data as float64!
        signal_pol.change_dtype('float32')
    signal_pol.compute()
signal_pol.save(datapath.with_stem(f"{datapath.stem}_pol"), overwrite=True)

In [ ]:
signal_pol = hs.load(datapath.with_stem(f"{datapath.stem}_pol"))

In [ ]:
#n = simulations.current_size #Keep the results from matching all the templates
n = 5 #If you only want to keep the five best results
signal_results = signal_pol.get_orientation(simulation=simulations, n_best=n, frac_keep=1.0)
signal_results

plot in polar space

In [ ]:
if True:
    n = signal_results.num_rows
    #n = 1 #If too crowded markers in the plot, uncomment this line to only plot the highest correlation scores
    
    signal_pol.plot(norm='symlog')
    signal_pol.add_marker(
        signal_results.to_single_phase_polar_markers(
            n_best=n,
            signal_axes=signal_pol.axes_manager.signal_axes
        )
    )

plot in kx, ky space

In [ ]:
signal.plot(cmap='viridis_r', norm='symlog')
signal.add_marker(signal_results.to_markers(annotate=True))

inspecting correlation scores

In [ ]:
signal_results.scatter('ipf', color_key=IPFColorKeyTSL(GaAs), axes_decor='off', tight_layout=True)

In [ ]:
signal_results.plot(interactive=True)

In [ ]:
pixel = [0,0]
correlations_i = signal_results.inav[pixel].data[:, 1]
template_indices_i = (signal_results.inav[pixel].data[:,0]).astype('int16')
beam_orientations_i = beam_orientations[template_indices_i]
euler_angles_i = beam_orientations_i.to_euler()

fig = plt.figure()
ax = fig.add_subplot(111, projection="ipf", symmetry=GaAs.point_group)
ax.scatter(beam_orientations_i, c=correlations_i, cmap='inferno')
ax.set_title(f'Correlation for pixel {pixel}')

In [ ]:
#fmt = 'html'
#fmt = 'simple'
#fmt = 'latex'
fmt = 'pipe' #For PHP Markdown

# Get the simulation of the template index
beam_direction, phase_index, simulation = simulations.get_simulation(template_indices_i[0]) #Returns a tuple with the beam direction, the phase index, and the simulation
phi1, theta, phi2 = beam_direction.to_euler(degrees=True).flatten() #Get the euler angles of the beam direction follwing the Bunge convention used in Orix.

#Create a nicely formatted table of the hkl indices.
table = tabulate(simulation.hkl, headers=['h', 'k', 'l'], floatfmt='.0f', tablefmt=fmt) 
print(f"(hkl) indices of reflections of euler angles ({phi1:.1f}, {theta:.1f}, {phi2:.1f}) of phase number {phase_index}:\n{table}")

In [ ]:
g1 = Miller(hkl=simulation.hkl[0], phase=GaAs)
g2 = Miller(hkl=simulation.hkl[2], phase=GaAs)
P = g1.cross(g2).round()
print(f"Zone axis of template is close to [{P.u[0]:.0f} {P.v[0]:.0f} {P.w[0]:.0f}]")

statistics

In [ ]:
results = pd.DataFrame({signal_results.column_names[i]:signal_results.isig[i,0].data.flatten() for i in range(len(signal_results.column_names))}) #Iterate over the columns and get the column names and the values.
results = results.sort_values('correlation', ascending=False) #Sort the data

#Plot the data
fig, ax = plt.subplots()
sb.histplot(data=results[['correlation']], bins=int(len(results)/10), ax=ax)
ax.set_xlabel('Correlation score')
print(results)

crystal map

In [ ]:
xmap = signal_results.to_crystal_map()
xmap.prop['index'] = signal_results.isig[0,0].data.flatten() #Get the template index
xmap.prop['ci'] = signal_results.isig[1,0].data.flatten() #Get the correlation scores of the best scores. Store it as "CI" - Confidence Index. This is a commonly use term for the matching scores in many EBSD solutions
xmap.prop['rotation'] = signal_results.isig[2,0].data.flatten() #Get the inplane rotation
xmap.prop['mirrored'] = signal_results.isig[3,0].data.flatten() #Get the mirrored flag
vbf = signal.get_integrated_intensity(hs.roi.CircleROI(0.0, 0.0, 0.1))
try:
    vbf.compute() #Load the lazy data into memory as you compute the virtual bright field
except Exception as e:
    print(e) #Ignore the error, but print it to let us know something happened.
xmap.prop['vbf'] = vbf.data.flatten() #Get the VBF data and add it as a property
save(datapath.with_stem(f"{datapath.stem}_xmap").with_suffix('.hdf5'), xmap, overwrite=True)
xmap

In [ ]:
xmap=load(datapath.with_stem(f"{datapath.stem}_xmap").with_suffix('.hdf5'))
GaAs = xmap.phases['GaAs']

In [ ]:
xmap_filtered = xmap[xmap.ci<10000] #treshold from correlation score histogram
oris = xmap_filtered.orientations
corrs = xmap_filtered.ci

oris.scatter('ipf', c=corrs, alpha=corrs/np.max(corrs), direction=Vector3d.xvector(), edgecolors=None, cmap='magma')
oris.scatter('ipf', c=corrs, alpha=corrs/np.max(corrs), direction=Vector3d.yvector(), edgecolors=None, cmap='magma')
oris.scatter('ipf', c=corrs, alpha=corrs/np.max(corrs), direction=Vector3d.zvector(), edgecolors=None, cmap='magma')

In [ ]:
key_x = IPFColorKeyTSL(GaAs.point_group, Vector3d.xvector())
key_y = IPFColorKeyTSL(GaAs.point_group, Vector3d.yvector())
key_z = IPFColorKeyTSL(GaAs.point_group, Vector3d.zvector())

#First plot the key and add the orientations as scatter points
with plt.rc_context({'font.size': 8}): #Temporary set fontsize
    ipf = key_x.plot(return_figure=True)
    xmap.orientations.scatter('ipf', figure=ipf, s=3, color='k', edgecolor='none')
    ipf.set_figheight(3) #Adjust figure height
    ipf.set_figwidth(3) #Adjust figure width

# Calculate the required sizes of the figues
nx, ny = xmap.shape
aspect_ratio = nx/ny
figure_width = 4
figure_kwargs={'figsize': (figure_width, figure_width*aspect_ratio)}

#Plot the IPF maps in individual figures
keys = [key_x, key_y, key_z]
titles = ['$X$', '$Y$', '$Z$']
with plt.rc_context({'font.size': 8}): #Temporary set fontsize
    for key, title in zip(keys, titles):
        figure = xmap.plot(key.orientation2color(xmap.orientations), figure_kwargs=figure_kwargs, remove_padding=True, return_figure=True, scalebar=False)
        figure.suptitle(title)

In [ ]:
keys = [IPFColorKeyTSL(GaAs.point_group, direction) for direction in (Vector3d.xvector(), Vector3d.yvector(), Vector3d.zvector())]

# Calculate the required sizes of the figues
nx, ny = xmap.shape
aspect_ratio = nx/ny
figure_width = 4
figure_kwargs={'figsize': (figure_width, figure_width*aspect_ratio)}

for key in keys:
    xmap.plot(key.orientation2color(xmap.orientations), overlay=xmap.ci, figure_kwargs=figure_kwargs, remove_padding=True, scalebar = False)

for key in keys:
    xmap.plot(key.orientation2color(xmap.orientations), overlay=xmap.vbf, figure_kwargs=figure_kwargs, remove_padding=True)

In [ ]:
xmap_filtered = xmap[xmap.ci>=10]
keys = [IPFColorKeyTSL(GaAs.point_group, direction) for direction in (Vector3d.xvector(), Vector3d.yvector(), Vector3d.zvector())]

# Calculate the required sizes of the figues
nx, ny = xmap_filtered.shape
aspect_ratio = nx/ny
figure_width = 4
figure_kwargs={'figsize': (figure_width, figure_width*aspect_ratio)}

for key in keys:
    xmap_filtered.plot(key.orientation2color(xmap_filtered.orientations), figure_kwargs=figure_kwargs, remove_padding=True, scalebar=False)

define rois for A, B, C and D

In [ ]:
roi_A = signal_results.inav[xA0:xA1, yA0:yA1]
roi_B = signal_results.inav[xB0:xB1, yB0:yB1]
roi_C = signal_results.inav[xC0:xC1, yC0:yC1]
roi_D = signal_results.inav[xD0:xD1, yD0:yD1]

In [ ]:
oris_A = roi_A.orientations
oris_B = roi_B.orientations

In [ ]:
signal_results.scatter('ipf', ...)